# NISAR Cloud Access

Stream NISAR granules from Earthdata Cloud and cache only bbox subsets for reuse.

Requires free Earthdata account: https://urs.earthdata.nasa.gov/


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import snowsar.utils.stream_utils as stream_utils

get_nisar_granule_name = stream_utils.get_nisar_granule_name
get_nisar_granule_urls = stream_utils.get_nisar_granule_urls
get_nisar_subset_cache_path = stream_utils.get_nisar_subset_cache_path
extract_gunw_layers_to_geotiff_bbox_streamed = stream_utils.extract_gunw_layers_to_geotiff_bbox_streamed
open_nisar_h5_stream = stream_utils.open_nisar_h5_stream
read_nisar_h5_bbox_cached = stream_utils.read_nisar_h5_bbox_cached
search_nisar_data = stream_utils.search_nisar_data
setup_asf_search_auth = stream_utils.setup_asf_search_auth
setup_earthaccess_auth = stream_utils.setup_earthaccess_auth
write_nisar_subset_geotiff = stream_utils.write_nisar_subset_geotiff

## Authenticate

Run once per session. If `~/.netrc` exists, credentials are read from it. Otherwise, these helpers prompt for Earthdata Login credentials.

In [ ]:
setup_earthaccess_auth(persist=True)
asf_session = setup_asf_search_auth()

## Search

Find NISAR products by bbox and date.

In [ ]:
search_bbox = (-120.5, 38.8, -120, 39.4)  # west, south, east, north

results = search_nisar_data(
    bbox=search_bbox,
    start_date="2026-01-01",
    end_date="2026-03-31",
    processing_level="GUNW",
    provider="asf_search", # or "earthaccess"
    flight_direction="DESCENDING",  # or "ASCENDING"
)

print(f"Found {len(results)} granules")
if results:
    print(f"First: {get_nisar_granule_name(results[0])}")
    urls = get_nisar_granule_urls(results[0])
    if urls:
        print(f"First URL: {urls[0]}")


## Inspect Available Frequencies And Polarizations

Open the first granule and list frequency groups plus available polarizations under `unwrappedInterferogram`.


In [ ]:
if results:
    with open_nisar_h5_stream(results[0], provider="earthaccess") as h5:
        grids = h5["science/LSAR/GUNW/grids"]
        frequencies = sorted(grids.keys())
        print("Available frequencies:", frequencies)

        for frequency in frequencies:
            unw_group = grids[frequency]["unwrappedInterferogram"]
            sub_groups = sorted(unw_group.keys())
            print(f"{frequency} sub-groups: {sub_groups}")
else:
    print("No results. Run the search cell first.")


## Stream First Result Subset For Visualization

Read only the requested bbox window without writing cache files. Visualize first, then choose whether to cache the arrays.


In [ ]:
if results:
    unw_path = "science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH/unwrappedPhase"
    coh_path = "science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH/coherenceMagnitude"

    # Stream data from the first returned URL
    unwrapped, xcoords, ycoords, epsg, unw_cache = read_nisar_h5_bbox_cached(
        results[0], unw_path, search_bbox,
    )
    coherence, _, _, _, coh_cache = read_nisar_h5_bbox_cached(
        results[0], coh_path, search_bbox,
    )

    plot_extent = [xcoords.min(), xcoords.max(), ycoords.min(), ycoords.max()]
    print(f"Subset bbox: {search_bbox}")
    print(f"Unwrapped subset shape: {unwrapped.shape}")
    print(f"Coherence subset shape: {coherence.shape}")
    print(f"EPSG: {epsg}")
else:
    print("No results. Adjust search params.")


## Visualize

Plot streamed data without downloading full file.

In [ ]:
if results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    im1 = ax1.imshow(
        unwrapped,
        cmap="RdBu_r",
        vmin=-10,
        vmax=10,
        extent=plot_extent,
        origin="upper",
    )
    ax1.set_title("Unwrapped Phase")
    plt.colorbar(im1, ax=ax1, label="rad")
    
    im2 = ax2.imshow(
        coherence,
        cmap="viridis",
        vmin=0,
        vmax=1,
        extent=plot_extent,
        origin="upper",
    )
    ax2.set_title("Coherence")
    plt.colorbar(im2, ax=ax2)
    
    plt.tight_layout()
    plt.show()

## Extract chosen layers for the first result for this bbox

Once you like the plotted bbox, stream and export the selected layers directly for that bbox without caching the full granule. Interpolated and derived layers require `sardem`, `GDAL`, `scipy`, and `rasterio`.


In [ ]:
selected_frequency = "A"
selected_pol = "HH"
bbox_extract_dir = Path("./outputs/nisar_bbox_layers")

layers = [
    "unwrappedPhase",
    "coherenceMagnitude",
    "ionospherePhaseScreen",
    "connectedComponents",
    "incidenceAngle",
    "localIncidenceAngle",
    "totalTroposphere",
    "losUnitVectorX",
    "losUnitVectorY",
    "elevationAngle",
    "slantRangeSolidEarthTidesPhase",
]


In [ ]:
if results:
    streamed_outputs = extract_gunw_layers_to_geotiff_bbox_streamed(
        results[0],
        search_bbox,
        bbox_extract_dir,
        frequency=selected_frequency,
        pol=selected_pol,
        layers=layers,
        provider="earthaccess",
        overwrite=False,
    )

    print(f"BBox layer directory: {bbox_extract_dir}")
    for layer_name, output_path in streamed_outputs.items():
        print(f"{layer_name}: {output_path}")
else:
    print("No results. Run the search cell first.")


## Batch Extract This Bbox For All Returned Granules

Use the same bbox, frequency, polarization, and layer list for every search result. This streams each granule in turn and writes one bbox-layer set per granule.


In [ ]:
batch_extract_dir = Path("./outputs/nisar_bbox_layers_batch")
max_batch_results = None  # set to an integer to limit processing
batch_max_retries = 2
batch_retry_delay = 1.0

selected_frequency = "A"
selected_pol = "HH"

layers = [
    "unwrappedPhase",
    "coherenceMagnitude",
    "ionospherePhaseScreen",
    "connectedComponents",
    "incidenceAngle",
    "localIncidenceAngle",
    "totalTroposphere",
    "losUnitVectorX",
    "losUnitVectorY",
    "elevationAngle",
    "slantRangeSolidEarthTidesPhase",
]

In [ ]:
if results:
    granules_to_extract = results[:max_batch_results] if max_batch_results is not None else results
    batch_outputs = {}
    batch_failures = {}

    print(f"Extracting {len(granules_to_extract)} granules to {batch_extract_dir}")
    for i, granule in enumerate(granules_to_extract, start=1):
        granule_name = get_nisar_granule_name(granule)
        print(f"[{i}/{len(granules_to_extract)}] {granule_name}")

        try:
            batch_outputs[granule_name] = extract_gunw_layers_to_geotiff_bbox_streamed(
                granule,
                search_bbox,
                batch_extract_dir,
                frequency=selected_frequency,
                pol=selected_pol,
                layers=layers,
                provider="earthaccess",
                overwrite=False,
                max_retries=batch_max_retries,
                retry_delay=batch_retry_delay,
            )
        except Exception as exc:
            batch_failures[granule_name] = str(exc)
            print(f"  failed: {exc}")

    print(f"Batch extraction complete: {len(batch_outputs)} succeeded, {len(batch_failures)} failed")
else:
    print("No results. Run the search cell first.")


In [ ]:
if results:
    if batch_outputs:
        first_name = next(iter(batch_outputs))
        print(first_name)
        batch_outputs[first_name]
    if batch_failures:
        print("Failures:")
        for granule_name, message in batch_failures.items():
            print(f"- {granule_name}: {message}")
